# Healthcare Appointment No-Show Prediction

## 01 - Data Understanding

This notebook focuses on understanding the raw healthcare appointment dataset before
performing data cleaning, exploratory data analysis, feature engineering, and model training.

### Objectives
- Load the raw dataset
- Understand the dataset structure
- Inspect data types
- Check missing values
- Check duplicate records
- Analyze categorical and numerical features
- Understand the target variable
- Identify potential data-quality issues
- Identify possible data leakage

In [3]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv('../data/raw/HealthCare.csv')

In [5]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [6]:
# check the dataset shape
df.shape

(110527, 14)

In [7]:
# checck column names
df.columns.tolist()

['PatientId',
 'AppointmentID',
 'Gender',
 'ScheduledDay',
 'AppointmentDay',
 'Age',
 'Neighbourhood',
 'Scholarship',
 'Hipertension',
 'Diabetes',
 'Alcoholism',
 'Handcap',
 'SMS_received',
 'No-show']

In [8]:
# info about data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


In [9]:
# dataset summary
summary = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes.astype(str),
    'Missing_values': df.isnull().sum(),
    'Missing_percentage': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique_Values': df.nunique()
})

summary

,Column,Data_Type,Missing_values,Missing_percentage,Unique_Values
PatientId,PatientId,float64,0,0.0,62299
AppointmentID,AppointmentID,int64,0,0.0,110527
Gender,Gender,object,0,0.0,2
ScheduledDay,ScheduledDay,object,0,0.0,103549
AppointmentDay,AppointmentDay,object,0,0.0,27
Age,Age,int64,0,0.0,104
Neighbourhood,Neighbourhood,object,0,0.0,81
Scholarship,Scholarship,int64,0,0.0,2
Hipertension,Hipertension,int64,0,0.0,2
Diabetes,Diabetes,int64,0,0.0,2


In [10]:
df['PatientId'].nunique()

62299

In [11]:
patient_counts = df['PatientId'].value_counts()

patient_counts.head(10)

PatientId
8.221459e+14    88
9.963767e+10    84
2.688613e+13    70
3.353478e+13    65
7.579746e+13    62
8.713749e+14    62
6.264199e+12    62
2.584244e+11    62
6.684488e+13    57
8.722785e+11    55
Name: count, dtype: int64

In [12]:
patient_counts.describe()

count    62299.000000
mean         1.774138
std          1.770324
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         88.000000
Name: count, dtype: float64

In [13]:
df['Handcap'].value_counts().sort_index()

Handcap
0    108286
1      2042
2       183
3        13
4         3
Name: count, dtype: int64

In [14]:
df['Age'].describe()

count    110527.000000
mean         37.088874
std          23.110205
min          -1.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         115.000000
Name: Age, dtype: float64

In [15]:
df['Age'].value_counts().sort_index().head(10)

Age
-1       1
 0    3539
 1    2273
 2    1618
 3    1513
 4    1299
 5    1489
 6    1521
 7    1427
 8    1424
Name: count, dtype: int64

In [16]:
df['Age'].value_counts().sort_index().tail(20)

Age
83     280
84     311
85     275
86     260
87     184
88     126
89     173
90     109
91      66
92      86
93      53
94      33
95      24
96      17
97      11
98       6
99       1
100      4
102      2
115      5
Name: count, dtype: int64

In [17]:
# duplicates

duplicate_count = df.duplicated().sum()

print('Duplicate rows:', duplicate_count)
print(
    'Duplicate percentage:',
    round(duplicate_count / len(df) * 100, 2),
    "%"
)

Duplicate rows: 0
Duplicate percentage: 0.0 %


In [18]:
# check the target 
df['No-show'].value_counts()

No-show
No     88208
Yes    22319
Name: count, dtype: int64

In [19]:
df['No-show'].value_counts(normalize = True).mul(100).round(2)

No-show
No     79.81
Yes    20.19
Name: proportion, dtype: float64

In [20]:
categorical_columns = [
    "Gender",
    "Scholarship",
    "Hipertension",
    "Diabetes",
    "Alcoholism",
    "Handcap",
    "SMS_received",
    "No-show"
]

for column in categorical_columns:
    print(f"\n{'=' * 50}")
    print(column)
    print(df[column].value_counts())


Gender
Gender
F    71840
M    38687
Name: count, dtype: int64

Scholarship
Scholarship
0    99666
1    10861
Name: count, dtype: int64

Hipertension
Hipertension
0    88726
1    21801
Name: count, dtype: int64

Diabetes
Diabetes
0    102584
1      7943
Name: count, dtype: int64

Alcoholism
Alcoholism
0    107167
1      3360
Name: count, dtype: int64

Handcap
Handcap
0    108286
1      2042
2       183
3        13
4         3
Name: count, dtype: int64

SMS_received
SMS_received
0    75045
1    35482
Name: count, dtype: int64

No-show
No-show
No     88208
Yes    22319
Name: count, dtype: int64


In [21]:
# checks the dates

print('ScheduledDay:')
print(df['ScheduledDay'].head())

print('\n AppointmentDay:')
print(df['AppointmentDay'].head())

ScheduledDay:
0    2016-04-29T18:38:08Z
1    2016-04-29T16:08:27Z
2    2016-04-29T16:19:04Z
3    2016-04-29T17:29:31Z
4    2016-04-29T16:07:23Z
Name: ScheduledDay, dtype: object

 AppointmentDay:
0    2016-04-29T00:00:00Z
1    2016-04-29T00:00:00Z
2    2016-04-29T00:00:00Z
3    2016-04-29T00:00:00Z
4    2016-04-29T00:00:00Z
Name: AppointmentDay, dtype: object


In [22]:
print("ScheduledDay dtype:", df["ScheduledDay"].dtype)
print("AppointmentDay dtype:", df["AppointmentDay"].dtype)

ScheduledDay dtype: object
AppointmentDay dtype: object


In [23]:
print("ScheduledDay range:")
print(df["ScheduledDay"].min())
print(df["ScheduledDay"].max())

print("\nAppointmentDay range:")
print(df["AppointmentDay"].min())
print(df["AppointmentDay"].max())

ScheduledDay range:
2015-11-10T07:13:56Z
2016-06-08T20:07:23Z

AppointmentDay range:
2016-04-29T00:00:00Z
2016-06-08T00:00:00Z


## Data Quality Findings

The raw dataset contains 110,527 appointment records and 14 columns.

Key findings:

- No missing values were identified.
- No duplicate rows were identified.
- The target variable is moderately imbalanced.
- 79.81% of appointments were attended.
- 20.19% of appointments were no-shows.
- PatientId is not unique and represents repeated appointments for some patients.
- AppointmentID is unique for every appointment.
- Handcap contains values from 0 to 4 and requires further investigation.
- ScheduledDay and AppointmentDay are currently stored as strings and require datetime conversion.
- PatientId and AppointmentID are identifiers and should not be used directly as ML features.

In [24]:
# Investigate on Handcap

In [25]:
pd.crosstab(
    df['Handcap'],
    df['No-show'],
    normalize = 'index'
).round(4) * 100

No-show,No,Yes
Handcap,,
0,79.76,20.24
1,82.08,17.92
2,79.78,20.22
3,76.92,23.08
4,66.67,33.33


In [26]:
# check Age

In [27]:
print('Minimum age:', df['Age'].min())
print('Maximum age:', df['Age'].max())
print('Mean age:', round(df['Age'].mean(),2))
print('Median Age:', df['Age'].median())

Minimum age: -1
Maximum age: 115
Mean age: 37.09
Median Age: 37.0


In [28]:
df['Age'].value_counts().sort_index().head(15)

Age
-1        1
 0     3539
 1     2273
 2     1618
 3     1513
 4     1299
 5     1489
 6     1521
 7     1427
 8     1424
 9     1372
 10    1274
 11    1195
 12    1092
 13    1103
Name: count, dtype: int64

In [29]:
df['Age'].value_counts().sort_index().tail(15)

Age
88     126
89     173
90     109
91      66
92      86
93      53
94      33
95      24
96      17
97      11
98       6
99       1
100      4
102      2
115      5
Name: count, dtype: int64

In [30]:
# invalid ages

In [32]:
print('Age < 0:', (df['Age'] < 0).sum())
print('Age == 0:', (df['Age'] == 0).sum())
print('Age > 100:', (df['Age'] > 0).sum())

Age < 0: 1
Age == 0: 3539
Age > 100: 106987


In [33]:
# convert dates on a copy

In [34]:
df_check = df.copy()

In [35]:
df_check['ScheduledDay'] = pd.to_datetime(
    df_check['ScheduledDay'],
    utc = True
)

df_check['AppointmentDay'] = pd.to_datetime(
    df_check['AppointmentDay'],
    utc = True
)

In [36]:
df_check[['ScheduledDay', 'AppointmentDay']].head()

,ScheduledDay,AppointmentDay
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00


In [37]:
df_check[['ScheduledDay', 'AppointmentDay']].dtypes

ScheduledDay      datetime64[ns, UTC]
AppointmentDay    datetime64[ns, UTC]
dtype: object

In [38]:
# create temporary waiting time 

In [39]:
df_check['WaitingDays'] = (
    df_check['AppointmentDay'].dt.normalize()
    - df_check['ScheduledDay'].dt.normalize()
).dt.days

In [40]:
df_check['WaitingDays'].describe()

count    110527.000000
mean         10.183702
std          15.254996
min          -6.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

In [41]:
print('Negative waiting days:', (df_check['WaitingDays'] < 0).sum())
print('Zero waiting days:', (df_check['WaitingDays'] == 0).sum())

Negative waiting days: 5
Zero waiting days: 38563


In [44]:
df_check[df_check['WaitingDays']<0][
    ['ScheduledDay', 'AppointmentDay', 'WaitingDays']
].head()

,ScheduledDay,AppointmentDay,WaitingDays
27033,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1
55226,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1
64175,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1
71533,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6
72362,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1


In [45]:
# scheduled day contains time 

In [47]:
df_check['ScheduledHour'] = df_check['ScheduledDay'].dt.hour

df_check['ScheduledHour'].value_counts().sort_index()

ScheduledHour
6      1578
7     19213
8     15349
9     12823
10    11056
11     8462
12     5422
13     9036
14     9127
15     8079
16     5542
17     2909
18     1340
19      488
20      100
21        3
Name: count, dtype: int64

In [48]:
# appointment weekday

In [49]:
df_check['AppointmentDayOfWeek'] = (
    df_check['AppointmentDay'].dt.day_name()
)

In [50]:
df_check['AppointmentDayOfWeek'].value_counts()

AppointmentDayOfWeek
Wednesday    25867
Tuesday      25640
Monday       22715
Friday       19019
Thursday     17247
Saturday        39
Name: count, dtype: int64

In [51]:
# check SMS 

In [52]:
pd.crosstab(
    df['SMS_received'],
    df['No-show'],
    normalize = 'index'
).round(4) * 100

No-show,No,Yes
SMS_received,,
0,83.30,16.70
1,72.43,27.57


In [53]:
# patient id investigation

In [54]:
print('Patients with more than one appointment')
print((patient_counts > 1).sum())

Patients with more than one appointment
24379


In [55]:
print('patients with 5+ appointments:')
print((patient_counts >= 5).sum())

patients with 5+ appointments:
2617


In [56]:
# Recheck the >100 values

In [57]:
age_over_100 = df[df['Age']>100]

print('Number of records with age > 100:', len(age_over_100))
print('\nAge distribution > 100:')
print(age_over_100['Age'].value_counts().sort_index())

Number of records with age > 100: 7

Age distribution > 100:
Age
102    2
115    5
Name: count, dtype: int64


In [58]:
negative_waiting = df_check[df_check['WaitingDays']<0]

negative_waiting[
    [
        'PatientId',
        'AppointmentID',
        'ScheduledDay',
        'AppointmentDay',
        'WaitingDays',
        'No-show'
    ]
]

,PatientId,AppointmentID,ScheduledDay,AppointmentDay,WaitingDays,No-show
27033,7.839273e+12,5679978,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1,Yes
55226,7.896294e+12,5715660,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1,Yes
64175,2.425226e+13,5664962,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1,Yes
71533,9.982316e+14,5686628,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6,Yes
72362,3.787482e+12,5655637,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1,Yes


In [59]:
df_check.groupby("WaitingDays")["No-show"].value_counts(normalize=True).head(20)

WaitingDays  No-show
-6           Yes        1.000000
-1           Yes        1.000000
 0           No         0.953531
             Yes        0.046469
 1           No         0.786495
             Yes        0.213505
 2           No         0.761784
             Yes        0.238216
 3           No         0.764706
             Yes        0.235294
 4           No         0.767297
             Yes        0.232703
 5           No         0.733903
             Yes        0.266097
 6           No         0.752044
             Yes        0.247956
 7           No         0.733184
             Yes        0.266816
 8           No         0.712693
             Yes        0.287307
Name: proportion, dtype: float64

## Initial Data Cleaning Rules

The following cleaning rules were identified during data-quality analysis:

1. PatientId and AppointmentID are identifiers and will not be used as model features.
2. Invalid age values (-1 and values above 100) will be treated as missing.
3. Age = 0 will be retained because it may represent infants.
4. ScheduledDay and AppointmentDay will be converted to datetime.
5. WaitingDays will be calculated using calendar dates.
6. Negative WaitingDays values will be treated as invalid.
7. No duplicate rows were found.
8. No missing values were found in the original dataset.
9. Handcap values will initially be retained.

df_clean = df.copy()

In [60]:
df_clean = df.copy()

In [61]:
df_clean["Age"] = df_clean["Age"].mask(
    (df_clean["Age"] < 0) | (df_clean["Age"] > 100)
)

In [62]:
df_clean["Age"].isna().sum()

np.int64(8)

In [63]:
df_clean["Age"].describe()

count    110519.000000
mean         37.084519
std          23.103165
min           0.000000
25%          18.000000
50%          37.000000
75%          55.000000
max         100.000000
Name: Age, dtype: float64

In [64]:
df_clean["ScheduledDay"] = pd.to_datetime(
    df_clean["ScheduledDay"],
    utc=True
)

df_clean["AppointmentDay"] = pd.to_datetime(
    df_clean["AppointmentDay"],
    utc=True
)

In [65]:
df_clean["WaitingDays"] = (
    df_clean["AppointmentDay"].dt.normalize()
    - df_clean["ScheduledDay"].dt.normalize()
).dt.days

In [66]:
df_clean["WaitingDays"].describe()

count    110527.000000
mean         10.183702
std          15.254996
min          -6.000000
25%           0.000000
50%           4.000000
75%          15.000000
max         179.000000
Name: WaitingDays, dtype: float64

In [68]:
df_clean["WaitingDays"] = df_clean["WaitingDays"].mask(
    df_clean["WaitingDays"] < 0
)

In [69]:
print(
    "Invalid WaitingDays:",
    df_clean["WaitingDays"].isna().sum()
)

Invalid WaitingDays: 5


In [70]:
print("Shape:", df_clean.shape)

print(
    "Missing Age:",
    df_clean["Age"].isna().sum()
)

print(
    "Missing WaitingDays:",
    df_clean["WaitingDays"].isna().sum()
)

print(
    "Duplicate rows:",
    df_clean.duplicated().sum()
)

Shape: (110527, 15)
Missing Age: 8
Missing WaitingDays: 5
Duplicate rows: 0
